In [120]:
from pathlib import Path

PATH_DAILY = "/home/gpu4080/ygdata/apple/joined_SAMPLE_GDD_2013_2024_APPLE_복숭아순나방.csv"
PATH_OBS   = "/home/gpu4080/ygdata/apple/peach_moth_obs_with_fuzzy_features_LONG2.csv"
GDD_DIR    = Path("/home/gpu4080/ygdata/apple/GDD_since_db")


In [121]:
import pandas as pd
import numpy as np

# 기상데이터 + 관측데이터 + GDD join 파일
# PATH_DAILY = "/Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/joined_SAMPLE_GDD_2013_2024_APPLE_복숭아순나방.csv"

# 관측(불규칙) + GR/fuzzy 피처 파일
# PATH_OBS   = "/Users/doyoung-gil/연구실/데이터/사과/peach_moth_obs_with_fuzzy_features_LONG2.csv"

# threshold (일단 3으로 시작)
THRESHOLD = 1


In [122]:
import pandas as pd
import numpy as np

# daily(joined) 읽기 + 표준 키 만들기
usecols_daily = [
    "지점ID", "일시",
    "일강수량(mm)", "최고기온(°C)", "최저기온(°C)", "평균기온(°C)",
    "평균 풍속(m/s)", "최대 풍속(m/s)",
    "DD10", "GDD10_cum",
    "(트랩)복숭아순나방-마리수",
]

daily = pd.read_csv(PATH_DAILY, usecols=usecols_daily)

daily["date"] = pd.to_datetime(daily["일시"], errors="coerce")
daily = daily.dropna(subset=["date"]).copy()

daily = daily.rename(columns={"지점ID": "site_id"})
daily["year"] = daily["date"].dt.year
daily["doy"]  = daily["date"].dt.dayofyear
daily = daily.sort_values(["site_id", "date"]).reset_index(drop=True)

# ---- daily 단계에서 기상 NaN 보간 ----
weather_cols = [
    "일강수량(mm)",
    "최고기온(°C)",
    "최저기온(°C)",
    "평균기온(°C)",
    "평균 풍속(m/s)",
    "최대 풍속(m/s)",
]

for c in weather_cols:
    daily[c] = pd.to_numeric(daily[c], errors="coerce")

out = []
for (site, year), sub in daily.groupby(["site_id", "year"], sort=False):
    sub = sub.sort_values("doy").copy()
    for c in weather_cols:
        sub[c] = sub[c].interpolate(limit_direction="both").ffill().bfill()
    sub["일강수량(mm)"] = sub["일강수량(mm)"].fillna(0.0)
    out.append(sub)

daily = pd.concat(out, ignore_index=True)

print("NaN rate after fill (daily):")
print(daily[weather_cols].isna().mean())
daily.head()


NaN rate after fill (daily):
일강수량(mm)      0.0
최고기온(°C)      0.0
최저기온(°C)      0.0
평균기온(°C)      0.0
평균 풍속(m/s)    0.0
최대 풍속(m/s)    0.0
dtype: float64


,site_id,일시,일강수량(mm),최고기온(°C),최대 풍속(m/s),최저기온(°C),평균 풍속(m/s),평균기온(°C),(트랩)복숭아순나방-마리수,DD10,GDD10_cum,date,year,doy
0,30523_63004,2013-01-01,0.0,1.6,17.7,-7.0,7.5,-2.7,NaN,0.0,0.0,2013-01-01,2013,1
1,30523_63004,2013-01-02,0.0,-3.7,13.6,-11.4,1.3,-9.2,NaN,0.0,0.0,2013-01-02,2013,2
2,30523_63004,2013-01-03,0.0,-8.0,9.5,-12.8,1.4,-10.5,NaN,0.0,0.0,2013-01-03,2013,3
3,30523_63004,2013-01-04,0.0,-5.8,8.5,-11.9,2.1,-8.7,NaN,0.0,0.0,2013-01-04,2013,4
4,30523_63004,2013-01-05,0.0,-1.2,7.5,-10.6,3.2,-5.9,NaN,0.0,0.0,2013-01-05,2013,5


In [123]:
# 휴면기 이후 계산한 GDD 추가

from pathlib import Path
import pandas as pd
import numpy as np
import re

# GDD_DIR = Path("/Users/doyoung-gil/연구실/데이터/사과/GDD_since_db")  

# daily 키 타입 맞추기
daily["site_id"] = daily["site_id"].astype(str)
daily["date"] = pd.to_datetime(daily["date"], errors="coerce")

dfs = []
bad_files = []

for fp in sorted(GDD_DIR.glob("*.csv")):
    # 파일명 예: site_32458_56647_GDD_timeseries.csv
    m = re.search(r"site_(\d+_\d+)", fp.name)
    if not m:
        bad_files.append(fp.name)
        continue

    site_id = m.group(1)  # "32458_56647"

    df = pd.read_csv(fp, usecols=["date", "GDD10_since_db"])
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["GDD10_since_db"] = pd.to_numeric(df["GDD10_since_db"], errors="coerce")
    df["site_id"] = site_id

    df = df.dropna(subset=["date"]).copy()
    dfs.append(df)

gdd_db = pd.concat(dfs, ignore_index=True)
gdd_db = gdd_db.drop_duplicates(["site_id", "date"], keep="last")

print("gdd_db shape:", gdd_db.shape)
if bad_files:
    print("filename parse failed (show first 5):", bad_files[:5])

# ✅ merge
daily = daily.merge(gdd_db, on=["site_id", "date"], how="left")

# 휴면 전에는 0이 자연스러움
daily["GDD10_since_db"] = daily["GDD10_since_db"].fillna(0.0)

print("after merge NaN rate:", daily["GDD10_since_db"].isna().mean())
daily[["site_id","date","GDD10_cum","GDD10_since_db"]].head()


gdd_db shape: (1661157, 3)
after merge NaN rate: 0.0


,site_id,date,GDD10_cum,GDD10_since_db
0,30523_63004,2013-01-01,0.0,0.0
1,30523_63004,2013-01-02,0.0,0.0
2,30523_63004,2013-01-03,0.0,0.0
3,30523_63004,2013-01-04,0.0,0.0
4,30523_63004,2013-01-05,0.0,0.0


In [124]:
daily.head()

,site_id,일시,일강수량(mm),최고기온(°C),최대 풍속(m/s),최저기온(°C),평균 풍속(m/s),평균기온(°C),(트랩)복숭아순나방-마리수,DD10,GDD10_cum,date,year,doy,GDD10_since_db
0,30523_63004,2013-01-01,0.0,1.6,17.7,-7.0,7.5,-2.7,NaN,0.0,0.0,2013-01-01,2013,1,0.0
1,30523_63004,2013-01-02,0.0,-3.7,13.6,-11.4,1.3,-9.2,NaN,0.0,0.0,2013-01-02,2013,2,0.0
2,30523_63004,2013-01-03,0.0,-8.0,9.5,-12.8,1.4,-10.5,NaN,0.0,0.0,2013-01-03,2013,3,0.0
3,30523_63004,2013-01-04,0.0,-5.8,8.5,-11.9,2.1,-8.7,NaN,0.0,0.0,2013-01-04,2013,4,0.0
4,30523_63004,2013-01-05,0.0,-1.2,7.5,-10.6,3.2,-5.9,NaN,0.0,0.0,2013-01-05,2013,5,0.0


In [125]:
# “일별 연속성” 체크 (site-year별 1~365가 있는지)
def check_daily_grid(daily_df, site, year):
    sub = daily_df[(daily_df["site_id"] == site) & (daily_df["year"] == year)]
    days = set(sub["doy"].astype(int).tolist())
    missing = [d for d in range(1, 366) if d not in days]
    return len(sub), len(missing), missing[:10]

tmp_key = daily[["site_id","year"]].drop_duplicates().iloc[0]
nrows, nmiss, miss10 = check_daily_grid(daily, tmp_key["site_id"], tmp_key["year"])
print("example:", tmp_key.to_dict(), "rows:", nrows, "missing:", nmiss, "miss10:", miss10)


example: {'site_id': '30523_63004', 'year': 2013} rows: 365 missing: 0 miss10: []


In [126]:
# =========================
# rolling/파생 feature 생성 (site-year 기준)
# =========================
daily = daily.sort_values(["site_id", "year", "doy"]).copy()
g = daily.groupby(["site_id", "year"], sort=False)

# 파생: 일교차
daily["trange"] = daily["최고기온(°C)"] - daily["최저기온(°C)"]

# rain rolling
daily["rain_7d_sum"]  = g["일강수량(mm)"].transform(lambda s: s.rolling(7,  min_periods=1).sum())
daily["rain_14d_sum"] = g["일강수량(mm)"].transform(lambda s: s.rolling(14, min_periods=1).sum())
daily["rain_7d_days"] = g["일강수량(mm)"].transform(lambda s: (s > 0).rolling(7, min_periods=1).sum())

# temp rolling
daily["tmean_7d_mean"] = g["평균기온(°C)"].transform(lambda s: s.rolling(7, min_periods=1).mean())
daily["tmax_7d_max"]   = g["최고기온(°C)"].transform(lambda s: s.rolling(7, min_periods=1).max())
daily["tmin_7d_min"]   = g["최저기온(°C)"].transform(lambda s: s.rolling(7, min_periods=1).min())

# DD/GDD rolling
daily["DD10_7d_sum"] = g["DD10"].transform(lambda s: s.rolling(7, min_periods=1).sum())

# trange rolling
daily["trange_7d_mean"] = g["trange"].transform(lambda s: s.rolling(7, min_periods=1).mean())
daily.head()


,site_id,일시,일강수량(mm),최고기온(°C),최대 풍속(m/s),최저기온(°C),평균 풍속(m/s),평균기온(°C),(트랩)복숭아순나방-마리수,DD10,...,GDD10_since_db,trange,rain_7d_sum,rain_14d_sum,rain_7d_days,tmean_7d_mean,tmax_7d_max,tmin_7d_min,DD10_7d_sum,trange_7d_mean
0,30523_63004,2013-01-01,0.0,1.6,17.7,-7.0,7.5,-2.7,NaN,0.0,...,0.0,8.6,0.0,0.0,0.0,-2.700000,1.6,-7.0,0.0,8.600000
1,30523_63004,2013-01-02,0.0,-3.7,13.6,-11.4,1.3,-9.2,NaN,0.0,...,0.0,7.7,0.0,0.0,0.0,-5.950000,1.6,-11.4,0.0,8.150000
2,30523_63004,2013-01-03,0.0,-8.0,9.5,-12.8,1.4,-10.5,NaN,0.0,...,0.0,4.8,0.0,0.0,0.0,-7.466667,1.6,-12.8,0.0,7.033333
3,30523_63004,2013-01-04,0.0,-5.8,8.5,-11.9,2.1,-8.7,NaN,0.0,...,0.0,6.1,0.0,0.0,0.0,-7.775000,1.6,-12.8,0.0,6.800000
4,30523_63004,2013-01-05,0.0,-1.2,7.5,-10.6,3.2,-5.9,NaN,0.0,...,0.0,9.4,0.0,0.0,0.0,-7.400000,1.6,-12.8,0.0,7.320000


In [127]:
# obs(LONG2) 읽기 + 관측일 정리
obs = pd.read_csv(PATH_OBS)

# 필수 컬럼 체크
need = ["site_id", "year", "obs_doy",
        "(트랩)복숭아순나방-마리수",
        "days_since_flowering",
        "days_since_growing_start",
        "is_growing"]
missing = [c for c in need if c not in obs.columns]
if missing:
    raise ValueError(f"Missing columns in LONG2: {missing}")

# obs_doy 숫자화
obs["obs_doy"] = pd.to_numeric(obs["obs_doy"], errors="coerce")
obs = obs.dropna(subset=["obs_doy"]).copy()
obs["obs_doy"] = obs["obs_doy"].astype(int)
for c in ["days_since_flowering", "days_since_growing_start", "is_growing"]:
    obs[c] = pd.to_numeric(obs[c], errors="coerce")


# 정렬
obs = obs.sort_values(["site_id", "year", "obs_doy"]).reset_index(drop=True)

print("obs shape:", obs.shape)
obs.head()


obs shape: (11274, 28)


,site_id,좌표-위도,좌표-경도,year,best_suitability,offset_days,window_idx,dormancy_months,growing_months,dormancy_start_date,...,복숭아순나방-피해과수/750과,복숭아순나방-피해과율(%),(트랩)복숭아순나방-마리수,days_since_flowering,days_since_growing_start,days_until_growing_end,is_growing,is_dormancy,GDD10_since_db,label_event
0,30523_63004,37.005913,126.55243,2014,0.949192,0,12,5,6.0,331,...,0.0,0.0,0.0,-26,-27,209.0,0,1,0.0,0
1,30523_63004,37.005913,126.55243,2014,0.949192,0,12,5,6.0,331,...,0.0,0.0,0.0,-12,-13,195.0,0,1,0.0,0
2,30523_63004,37.005913,126.55243,2014,0.949192,0,12,5,6.0,331,...,0.0,0.0,0.0,5,4,178.0,1,0,23.8,0
3,30523_63004,37.005913,126.55243,2014,0.949192,0,12,5,6.0,331,...,0.0,0.0,0.0,40,39,143.0,1,0,262.8,0
4,30523_63004,37.005913,126.55243,2014,0.949192,0,12,5,6.0,331,...,0.0,0.0,0.0,40,39,143.0,1,0,262.8,0


In [128]:
print("obs shape:", obs.shape)
print("obs columns:", obs.columns.tolist()[:50])
print("obs_date na:", obs["obs_date"].isna().mean() if "obs_date" in obs.columns else "NO obs_date col")

# group key가 진짜 있는지
print("has site_id:", "site_id" in obs.columns)
print("has year:", "year" in obs.columns)

# 트랩 컬럼 확인(비슷한 이름 찾기)
cand = [c for c in obs.columns if "복숭아" in c or "트랩" in c or "마리" in c]
print("candidate count cols:", cand)


obs shape: (11274, 28)
obs columns: ['site_id', '좌표-위도', '좌표-경도', 'year', 'best_suitability', 'offset_days', 'window_idx', 'dormancy_months', 'growing_months', 'dormancy_start_date', 'dormancy_end_date', 'growing_start_date', 'growing_end_date', 'flowering_date', '조사일자(YYYYMMDD)', 'obs_doy', '복숭아순나방-마리수/640잎', '복숭아순나방-피해잎률(%)', '복숭아순나방-피해과수/750과', '복숭아순나방-피해과율(%)', '(트랩)복숭아순나방-마리수', 'days_since_flowering', 'days_since_growing_start', 'days_until_growing_end', 'is_growing', 'is_dormancy', 'GDD10_since_db', 'label_event']
obs_date na: NO obs_date col
has site_id: True
has year: True
candidate count cols: ['복숭아순나방-마리수/640잎', '복숭아순나방-피해잎률(%)', '복숭아순나방-피해과수/750과', '복숭아순나방-피해과율(%)', '(트랩)복숭아순나방-마리수']


In [129]:
# 같은 날 여러 번 관측이 있으면 (트랩)복숭아순나방-마리수의 최대값으로 대표
THRESHOLD = 1
count_col = "(트랩)복숭아순나방-마리수"

obs2 = obs.copy()

obs2["obs_doy"] = pd.to_numeric(obs2["obs_doy"], errors="coerce")
obs2[count_col] = pd.to_numeric(obs2[count_col], errors="coerce")

# ✅ 미관측 처리: count가 NaN이면 제거
obs2 = obs2.dropna(subset=["obs_doy", count_col]).copy()
obs2["obs_doy"] = obs2["obs_doy"].astype(int)

# 1일 1레코드(그날 max) — 남는 건 “실제로 관측값이 있는 날”만
obs2 = (obs2.groupby(["site_id","year","obs_doy"], as_index=False)[count_col].max())


print("obs2 shape:", obs2.shape)
obs2.head()

obs2 shape: (9349, 4)


,site_id,year,obs_doy,(트랩)복숭아순나방-마리수
0,30523_63004,2014,90,0.0
1,30523_63004,2014,104,0.0
2,30523_63004,2014,121,0.0
3,30523_63004,2014,156,0.0
4,30523_63004,2014,167,0.0


In [130]:
def build_interval_labels_from_doy(obs_df, threshold=1.0, count_col=count_col,
                                  season_start_doy=1, season_end_doy=365):
    rows = []
    for (site, year), sub in obs_df.groupby(["site_id","year"], sort=False):
        sub = sub.sort_values("obs_doy")

        y = sub[count_col].to_numpy()          # ✅ fillna(0) 제거
        t = sub["obs_doy"].to_numpy().astype(int)
        above = y >= threshold

        if above.any():
            idx_R = int(np.argmax(above))
            R_doy = int(t[idx_R])

            if idx_R == 0:
                censor_type = "left"
                L_doy = season_start_doy
            else:
                idx_Ls = np.where(~above[:idx_R])[0]
                if len(idx_Ls) == 0:
                    censor_type = "left"
                    L_doy = season_start_doy
                else:
                    censor_type = "interval"
                    L_doy = int(t[idx_Ls[-1]])

            rows.append({"site_id": site, "year": int(year), "censor_type": censor_type,
                         "L_doy": int(L_doy), "R_doy": int(R_doy), "threshold": threshold})
        else:
            rows.append({"site_id": site, "year": int(year), "censor_type": "right",
                         "L_doy": season_start_doy, "R_doy": season_end_doy, "threshold": threshold})

    return pd.DataFrame(rows)

labels = build_interval_labels_from_doy(obs2, threshold=THRESHOLD, count_col=count_col)
print(labels["censor_type"].value_counts())
labels.head()

censor_type
interval    350
left        346
right       271
Name: count, dtype: int64


,site_id,year,censor_type,L_doy,R_doy,threshold
0,30523_63004,2014,interval,183,197,1
1,30523_63004,2015,right,1,365,1
2,30523_63004,2016,right,1,365,1
3,30523_63004,2017,interval,121,136,1
4,30523_63004,2018,right,1,365,1


In [131]:
# interval 길이 큰 애들 제거 (labels 단계에서)
DOY_START, DOY_END = 60, 300
T = DOY_END - DOY_START + 1

MAX_GAP = 30   # 먼저 30으로 시작 (원하면 14/21로도 테스트)

# labels: site_id, year, censor_type, L_doy, R_doy 있다고 가정
labels_f = labels.copy()

# 시즌 범위로 클램프해서 gap 계산
labels_f["L_cl"] = labels_f["L_doy"].clip(DOY_START, DOY_END)
labels_f["R_cl"] = labels_f["R_doy"].clip(DOY_START, DOY_END)
labels_f["gap"]  = labels_f["R_cl"] - labels_f["L_cl"]

print("interval gap max(clipped):", labels_f.loc[labels_f["censor_type"]=="interval", "gap"].max())

# interval만 필터링 (right/left는 그대로 유지)
before = len(labels_f)
labels_f = labels_f[(labels_f["censor_type"] != "interval") | (labels_f["gap"] <= MAX_GAP)].copy()
after = len(labels_f)

print(f"labels kept: {after}/{before}  (dropped {before-after})")
print(labels_f["censor_type"].value_counts())

labels = labels_f.drop(columns=["L_cl","R_cl","gap"], errors="ignore")


interval gap max(clipped): 76
labels kept: 946/967  (dropped 21)
censor_type
left        346
interval    329
right       271
Name: count, dtype: int64


In [132]:
# left-censored 데이터 처리
# 관측 프로세스 feature 만들기: first_obs_doy, n_obs, max_gap
# A) obs2로 site-year별 관측 메타 만들기
import pandas as pd
import numpy as np

DOY_START, DOY_END = 60, 300

# obs2는 ["site_id","year","obs_doy","(트랩)복숭아순나방-마리수"] 형태라고 가정
obs2_season = obs2[(obs2["obs_doy"] >= DOY_START) & (obs2["obs_doy"] <= DOY_END)].copy()
obs2_season = obs2_season.sort_values(["site_id","year","obs_doy"])

def make_obs_meta(df):
    rows = []
    for (site, year), sub in df.groupby(["site_id","year"], sort=False):
        d = sub["obs_doy"].to_numpy()
        d = np.sort(d)
        n_obs = len(d)
        first_obs = int(d[0])
        # 관측 간격 최대
        if n_obs >= 2:
            max_gap = int(np.max(np.diff(d)))
        else:
            max_gap = int(DOY_END - DOY_START)  # 관측 1번뿐이면 매우 불확실하다고 크게
        rows.append({
            "site_id": site,
            "year": int(year),
            "first_obs_doy": first_obs,
            "n_obs": int(n_obs),
            "max_gap": int(max_gap),
        })
    return pd.DataFrame(rows)

obs_meta = make_obs_meta(obs2_season)

# 시즌 좌표(1..T)로도 만들어두면 모델에 더 직관적임
T = DOY_END - DOY_START + 1
obs_meta["first_obs_season"] = (obs_meta["first_obs_doy"] - DOY_START + 1).clip(1, T)

print(obs_meta.head())
print(obs_meta[["n_obs","max_gap","first_obs_doy"]].describe())


       site_id  year  first_obs_doy  n_obs  max_gap  first_obs_season
0  30523_63004  2014             90     13       35                31
1  30523_63004  2015             91     14       17                32
2  30523_63004  2016             92     15       22                33
3  30523_63004  2017             93     15       20                34
4  30523_63004  2018            155      1      240                96
            n_obs     max_gap  first_obs_doy
count  967.000000  967.000000     967.000000
mean     9.537746   35.755946     121.455016
std      4.147435   48.582158      33.182752
min      1.000000    1.000000      89.000000
25%      7.000000   18.000000      92.000000
50%     10.000000   20.000000     121.000000
75%     13.000000   30.000000     136.000000
max     15.000000  240.000000     295.000000


In [133]:
# =========================
# daily(연속 일별) + labels(구간 라벨) + obs_meta merge (완성형)
# 전제: daily(보간+rolling 완료), labels, obs_meta(시즌 기반 생성) 준비되어 있어야 함
# =========================

# 0) 타입 통일 (merge 매칭 깨짐 방지)
daily["site_id"]  = daily["site_id"].astype(str)
daily["year"]     = daily["year"].astype(int)
labels["site_id"] = labels["site_id"].astype(str)
labels["year"]    = labels["year"].astype(int)
obs_meta["site_id"] = obs_meta["site_id"].astype(str)
obs_meta["year"]    = obs_meta["year"].astype(int)

# 1) Transformer 입력 feature들: 원본 + rolling/파생
feature_cols = [
    # 원본(기존)
    "일강수량(mm)",
    "최고기온(°C)",
    "최저기온(°C)",
    "평균기온(°C)",
    "평균 풍속(m/s)",
    "최대 풍속(m/s)",
    "DD10",
    "GDD10_since_db",

    # rolling/파생(추가)
    "rain_7d_sum", "rain_14d_sum", "rain_7d_days",
    "tmean_7d_mean", "tmax_7d_max", "tmin_7d_min",
    "DD10_7d_sum",
    "trange", "trange_7d_mean",
]

# 2) daily에서 필요한 컬럼만 추출
daily_feat = daily[["site_id", "year", "doy", "date"] + feature_cols].copy()

# 3) labels 붙이기 (site-year 기준)
train_df = daily_feat.merge(labels, on=["site_id", "year"], how="inner")

# 4) obs_meta 붙이기 (관측 프로세스 feature)
# obs_meta columns 예: ["site_id","year","first_obs_season","n_obs","max_gap"] (+ first_obs_doy 등)
train_df2 = train_df.merge(
    obs_meta[["site_id", "year", "first_obs_season", "n_obs", "max_gap"]],
    on=["site_id", "year"],
    how="left"
)

# 5) obs_meta 없는 케이스 기본값 채우기
# (시즌 길이 T는 이미 위에서 T=DOY_END-DOY_START+1로 정의돼 있어야 함)
train_df2["first_obs_season"] = train_df2["first_obs_season"].fillna(1).astype(int)
train_df2["n_obs"]            = train_df2["n_obs"].fillna(0).astype(int)
train_df2["max_gap"]          = train_df2["max_gap"].fillna(T).astype(int)

print("train_df2 shape:", train_df2.shape)
train_df2.head()

train_df2 shape: (345571, 28)


,site_id,year,doy,date,일강수량(mm),최고기온(°C),최저기온(°C),평균기온(°C),평균 풍속(m/s),최대 풍속(m/s),...,DD10_7d_sum,trange,trange_7d_mean,censor_type,L_doy,R_doy,threshold,first_obs_season,n_obs,max_gap
0,30523_63004,2014,1,2014-01-01,0.0,7.3,3.1,5.4,9.1,17.3,...,0.0,4.2,4.200000,interval,183,197,1,31,13,35
1,30523_63004,2014,2,2014-01-02,0.0,4.7,1.0,2.8,1.5,6.4,...,0.0,3.7,3.950000,interval,183,197,1,31,13,35
2,30523_63004,2014,3,2014-01-03,0.0,7.7,0.1,2.8,2.7,7.9,...,0.0,7.6,5.166667,interval,183,197,1,31,13,35
3,30523_63004,2014,4,2014-01-04,0.0,2.1,-1.0,0.5,0.9,8.4,...,0.0,3.1,4.650000,interval,183,197,1,31,13,35
4,30523_63004,2014,5,2014-01-05,0.0,3.0,-1.2,0.6,1.1,7.2,...,0.0,4.2,4.560000,interval,183,197,1,31,13,35


In [134]:
# site static(lat/lon) 추가 (train_df2 단계에서)
site_static = (
    obs[["site_id", "좌표-위도", "좌표-경도"]]
    .drop_duplicates("site_id")
    .copy()
)


site_static["좌표-위도"] = pd.to_numeric(site_static["좌표-위도"], errors="coerce")
site_static["좌표-경도"] = pd.to_numeric(site_static["좌표-경도"], errors="coerce")

train_df2 = train_df2.merge(site_static, on="site_id", how="left")

# 결측이면 전체 평균으로(간단 처리)
train_df2["좌표-위도"] = train_df2["좌표-위도"].fillna(train_df2["좌표-위도"].mean())
train_df2["좌표-경도"] = train_df2["좌표-경도"].fillna(train_df2["좌표-경도"].mean())

feature_cols = feature_cols + ["좌표-위도", "좌표-경도"]
print("new D_in =", len(feature_cols))

new D_in = 19


In [135]:
# =========================
# [세트A] LONG2 phenology 컬럼을 daily(train_df2)에 doy 기준으로 merge + ffill
# =========================
# obs에는 days_since_flowering, days_since_growing_start, is_growing 컬럼이 있어야 함

pheno_cols = ["site_id","year","obs_doy","days_since_flowering","days_since_growing_start","is_growing"]
pheno = obs[pheno_cols].copy()

pheno["site_id"] = pheno["site_id"].astype(str)
pheno["year"] = pheno["year"].astype(int)
pheno["obs_doy"] = pd.to_numeric(pheno["obs_doy"], errors="coerce").astype("Int64")

for c in ["days_since_flowering","days_since_growing_start","is_growing"]:
    pheno[c] = pd.to_numeric(pheno[c], errors="coerce")

pheno = pheno.dropna(subset=["obs_doy"]).copy()
pheno["doy"] = pheno["obs_doy"].astype(int)
pheno = pheno.drop(columns=["obs_doy"])

# ✅ [추가] 하루 1레코드로 압축해서 중복 merge 방지
pheno = (pheno.sort_values(["site_id","year","doy"])
              .groupby(["site_id","year","doy"], as_index=False)
              .agg({
                  "days_since_flowering": "last",
                  "days_since_growing_start": "last",
                  "is_growing": "max",   # 0/1이면 max가 자연스러움
              }))

# 1) doy 기준 left-merge (이게 빠지면 안 됨)
train_df2 = train_df2.merge(pheno, on=["site_id","year","doy"], how="left")

# 2) 정렬
train_df2 = train_df2.sort_values(["site_id","year","doy"]).copy()

# 3) groupby ffill (apply 금지)
cols_fill = ["days_since_flowering", "days_since_growing_start", "is_growing"]
train_df2[cols_fill] = (
    train_df2.groupby(["site_id","year"], sort=False)[cols_fill].ffill()
)

# 4) 남은 결측 처리
train_df2["days_since_flowering"] = train_df2["days_since_flowering"].fillna(0.0)
train_df2["days_since_growing_start"] = train_df2["days_since_growing_start"].fillna(0.0)
train_df2["is_growing"] = train_df2["is_growing"].fillna(0.0)




In [136]:
# site-year별로 DOY가 연속인지 점검
def missing_doys_one_group(df, site, year):
    sub = df[(df["site_id"] == site) & (df["year"] == year)].sort_values("doy")
    have = set(sub["doy"].astype(int).tolist())
    miss = [d for d in range(1, 366) if d not in have]
    return len(sub), len(miss), miss[:20]

# 랜덤하게 하나 찍어서 확인
one = train_df[["site_id","year"]].drop_duplicates().sample(1, random_state=0).iloc[0]
nrows, nmiss, miss20 = missing_doys_one_group(train_df, one["site_id"], one["year"])
print("example group:", one.to_dict(), "rows:", nrows, "missing:", nmiss, "miss20:", miss20)


example group: {'site_id': '37730_57753', 'year': 2021} rows: 365 missing: 0 miss20: []


In [137]:
print("GDD10_since_db in train_df2?", "GDD10_since_db" in train_df2.columns)


GDD10_since_db in train_df2? True


In [138]:
# feature_cols에 추가 
feature_cols = feature_cols + ["first_obs_season", "n_obs", "max_gap"]
print("new D_in =", len(feature_cols))

new D_in = 22


In [139]:
print("site_id/year kept?", "site_id" in train_df2.columns, "year" in train_df2.columns)


site_id/year kept? True True


In [140]:
# (이미 train_df2 만들어져 있고, train_df_season도 만든 상태라고 가정)

# ---------- 여기서 run 선택 ----------
RUN = 5  

base = [
    "일강수량(mm)", "최고기온(°C)", "최저기온(°C)", "평균기온(°C)",
    "평균 풍속(m/s)", "최대 풍속(m/s)",
    "DD10", "GDD10_since_db",
]
meta = ["first_obs_season", "n_obs", "max_gap", "좌표-위도", "좌표-경도"]
roll = [
    "rain_7d_sum", "rain_14d_sum", "rain_7d_days",
    "tmean_7d_mean", "tmax_7d_max", "tmin_7d_min",
    "DD10_7d_sum",
    "trange", "trange_7d_mean",
]

if RUN == 1:
    feature_cols = base
elif RUN == 2:
    feature_cols = base + meta
elif RUN == 3:
    feature_cols = base + meta + roll
elif RUN == 4:
    feature_cols = meta + roll + ["GDD10_since_db"]
elif RUN == 5:
    feature_cols = meta + roll + ["GDD10_since_db",
                                  "days_since_flowering","days_since_growing_start","is_growing"] 
else:
    raise ValueError("다시")

print("RUN", RUN, "| D_in =", len(feature_cols))

FEATURE_COLS_RUN = feature_cols.copy()
print("FEATURE_COLS_RUN =", len(FEATURE_COLS_RUN))


RUN 5 | D_in = 18
FEATURE_COLS_RUN = 18


In [141]:
# 시즌 윈도우로 변경해서(예: DOY 60~300) 텐서 만들기

DOY_START, DOY_END = 60, 300
train_df_season = train_df2[(train_df2["doy"] >= DOY_START) & (train_df2["doy"] <= DOY_END)].copy()

# ✅ 라벨도 시즌 범위로 클램프 (중요)
train_df_season["L_doy"] = train_df_season["L_doy"].clip(lower=DOY_START, upper=DOY_END)
train_df_season["R_doy"] = train_df_season["R_doy"].clip(lower=DOY_START, upper=DOY_END)

# ✅ 시즌 길이 T
T = DOY_END - DOY_START + 1
print("Season T =", T)

# ✅ 이제 samples 만들기 (build_samples를 시즌용으로 약간 수정)
def build_samples_season(df, feature_cols, DOY_START=60, DOY_END=300):
    T = DOY_END - DOY_START + 1
    samples = []
    for (site, year), sub in df.groupby(["site_id","year"], sort=False):
        sub = sub.sort_values("doy")
        # 시즌 구간이 정확히 T개인지 확인
        if len(sub) != T:
            # 혹시 빠진 day가 있으면 에러 (현재는 missing=0이라 OK일 가능성 큼)
            continue

        X = sub[feature_cols].to_numpy(dtype=np.float32)

        # 라벨을 시즌 좌표(1..T)로 변환
        L = int(sub["L_doy"].iloc[0]) - DOY_START + 1
        R = int(sub["R_doy"].iloc[0]) - DOY_START + 1
        ctype = str(sub["censor_type"].iloc[0])

        # clamp
        L = min(max(L, 1), T)
        R = min(max(R, 1), T)

        samples.append({"site_id": site, "year": int(year), "X": X, "L": L, "R": R, "censor_type": ctype})
    return samples

samples = build_samples_season(train_df_season, FEATURE_COLS_RUN, DOY_START, DOY_END)


Season T = 241


In [142]:
# =========================
# Step7-compact: 정의 + split/norm + Dataset + DataLoader + 모델 정의까지 한 번에
# (학습 루프/early stopping은 별도 셀에서)
# 전제: samples(list[dict])가 이미 만들어져 있어야 함
# =========================
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ---- 1) split (site 기준) ----
def split_by_site(samples, val_frac=0.1, test_frac=0.1, seed=42):
    rng = np.random.default_rng(seed)
    sites = sorted(list({s["site_id"] for s in samples}))
    rng.shuffle(sites)

    n = len(sites)
    n_test = int(n * test_frac)
    n_val  = int(n * val_frac)

    test_sites  = set(sites[:n_test])
    val_sites   = set(sites[n_test:n_test+n_val])
    train_sites = set(sites[n_test+n_val:])

    train = [s for s in samples if s["site_id"] in train_sites]
    val   = [s for s in samples if s["site_id"] in val_sites]
    test  = [s for s in samples if s["site_id"] in test_sites]
    return train, val, test

train_samples, val_samples, test_samples = split_by_site(samples, val_frac=0.1, test_frac=0.1, seed=42)
print("split:", len(train_samples), len(val_samples), len(test_samples))

# ---- 2) norm stats (train만) ----
def compute_norm_stats(samples):
    X_all = np.concatenate([s["X"][None, :, :] for s in samples], axis=0)  # (N,T,D)
    mean = X_all.reshape(-1, X_all.shape[-1]).mean(axis=0)
    std  = X_all.reshape(-1, X_all.shape[-1]).std(axis=0)
    std = np.where(std < 1e-8, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32)

x_mean, x_std = compute_norm_stats(train_samples)
print("x_mean:", x_mean)
print("x_std :", x_std)

# ---- 3) Dataset ----
CTYPE2ID = {"interval": 0, "right": 1, "left": 2}

class IntervalEventDataset(Dataset):
    def __init__(self, samples, mean, std):
        self.samples = samples
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        X = (s["X"] - self.mean) / self.std
        X = torch.from_numpy(X).float()  # (T,D)
        L = torch.tensor(int(s["L"]), dtype=torch.long)
        R = torch.tensor(int(s["R"]), dtype=torch.long)
        c = torch.tensor(CTYPE2ID[str(s["censor_type"])], dtype=torch.long)
        return X, L, R, c

train_ds = IntervalEventDataset(train_samples, x_mean, x_std)
val_ds   = IntervalEventDataset(val_samples,   x_mean, x_std)
test_ds  = IntervalEventDataset(test_samples,  x_mean, x_std)

# ---- 4) DataLoader ----
train_loader = DataLoader(train_ds, batch_size=64,  shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds,  batch_size=128, shuffle=False, drop_last=False)

# ---- 5) Model 정의 ----
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=400):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class HazardTransformer(nn.Module):
    def __init__(self, d_in, d_model=64, nhead=4, num_layers=3, dropout=0.2, max_len=400):
        super().__init__()
        self.in_proj = nn.Linear(d_in, d_model)
        self.pos = PositionalEncoding(d_model, max_len=max_len)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4*d_model,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1)
        )

    def forward(self, X):
        h = self.in_proj(X)
        h = self.pos(h)
        h = self.encoder(h)
        logits = self.head(h).squeeze(-1)          # (B,T)
        return torch.sigmoid(logits).clamp(1e-6, 1-1e-6)

# ---- 6) sanity ----
T = int(train_samples[0]["X"].shape[0])
D = int(train_samples[0]["X"].shape[1])
print("T =", T, "| D =", D)
print("one batch shapes:", next(iter(train_loader))[0].shape)  # (B,T,D)


device: cuda
split: 760 99 87
x_mean: [6.2830265e+01 9.4894733e+00 3.3677631e+01 3.6440590e+01 1.2786379e+02
 3.2421810e+01 6.4697350e+01 2.1982203e+00 1.7470575e+01 2.7374197e+01
 9.0464973e+00 5.7711727e+01 1.1800054e+01 1.1795290e+01 7.2455835e+02
 5.2725777e+01 5.1968960e+01 6.5245688e-01]
x_std : [3.2437717e+01 4.0729032e+00 4.5422413e+01 6.6701281e-01 7.4314010e-01
 5.1100304e+01 8.1090439e+01 1.6502699e+00 6.8778200e+00 5.8132119e+00
 8.6990576e+00 3.9596451e+01 4.7894478e+00 3.2883205e+00 7.2360492e+02
 7.6797440e+01 7.6633820e+01 4.7638243e-01]
T = 241 | D = 18
one batch shapes: torch.Size([64, 241, 18])


In [143]:
# =========================================================
# [함수 정의 셀] 학습용 + 평가용 함수들 (중복 제거, 완성본)
# - early stopping 전에 한 번만 실행
# 전제: device가 이미 정의돼 있어야 함
# =========================================================
import numpy as np
import torch

# -------------------------
# Loss weights (run3 세팅)
# -------------------------
W_INTERVAL = 1.0
W_RIGHT    = 0.5
W_LEFT     = 0.5

# -------------------------
# (A) Train: per-sample NLL + weighted loss + epoch runner
# -------------------------
def interval_nll_per_sample(hazard, L, R, ctype, Tend):
    """
    hazard: (B,T)
    L,R,ctype: (B,)
    returns: (B,) nll
    """
    B, Tcur = hazard.shape
    assert Tcur == Tend

    log_surv_terms = torch.log1p(-hazard)          # (B,T)
    logS = torch.cumsum(log_surv_terms, dim=1)     # logS[:, t-1] = log S_t

    L = torch.clamp(L, 1, Tend)
    R = torch.clamp(R, 1, Tend)
    idxL = (L - 1).long()
    idxR = (R - 1).long()

    logS_L = logS.gather(1, idxL.view(-1,1)).squeeze(1)
    logS_R = logS.gather(1, idxR.view(-1,1)).squeeze(1)

    nll = torch.zeros(B, device=hazard.device)

    mi = (ctype == 0)   # interval
    mr = (ctype == 1)   # right
    ml = (ctype == 2)   # left

    # interval: -log(S_L - S_R)
    if mi.any():
        a = logS_L[mi]
        b = logS_R[mi]
        a = torch.maximum(a, b + 1e-8)
        nll[mi] = -(a + torch.log1p(-torch.exp(b - a)))

    # right: -log(S_T)
    if mr.any():
        nll[mr] = -logS[:, -1][mr]

    # left: -log(1 - S_R)
    if ml.any():
        a = logS_R[ml]
        cutoff = -0.6931471805599453  # log(0.5)
        out = torch.empty_like(a)
        m = a < cutoff
        out[m]  = torch.log1p(-torch.exp(a[m]))
        out[~m] = torch.log(-torch.expm1(a[~m]))
        nll[ml] = -out

    return nll

def weighted_loss_from_ctype(nll_vec, ctype):
    w = torch.ones_like(nll_vec)
    w = torch.where(ctype==0, w.new_tensor(W_INTERVAL), w)
    w = torch.where(ctype==1, w.new_tensor(W_RIGHT),    w)
    w = torch.where(ctype==2, w.new_tensor(W_LEFT),     w)
    return (w * nll_vec).mean()

def run_epoch_weighted(model, opt, loader, Tend, train=True):
    model.train(train)
    total, n = 0.0, 0
    for X, L, R, ctype in loader:
        X = X.to(device); L = L.to(device); R = R.to(device); ctype = ctype.to(device)

        hazard = model(X)
        nll_vec = interval_nll_per_sample(hazard, L, R, ctype, Tend=Tend)
        loss = weighted_loss_from_ctype(nll_vec, ctype)

        if train:
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        total += loss.item() * X.size(0)
        n += X.size(0)
    return total / max(n, 1)

@torch.no_grad()
def eval_nll_model(model, loader, Tend):
    model.eval()
    total, n = 0.0, 0
    for X, L, R, ctype in loader:
        X = X.to(device); L = L.to(device); R = R.to(device); ctype = ctype.to(device)
        hazard = model(X)
        nll_vec = interval_nll_per_sample(hazard, L, R, ctype, Tend=Tend)
        loss = weighted_loss_from_ctype(nll_vec, ctype)
        total += loss.item() * X.size(0)
        n += X.size(0)
    return total / max(n, 1)

# -------------------------
# (B) Eval: point_cov / MAE / mass / IoU@80 / Recall / Precision
# -------------------------
CTYPE_INTERVAL = 0  # interval=0, right=1, left=2

@torch.no_grad()
def hazard_to_pmf_cdf_logS(hazard):
    B, T = hazard.shape
    logS = torch.cumsum(torch.log1p(-hazard), dim=1)  # log S_t
    S_prev = torch.cat([torch.ones(B,1,device=hazard.device), torch.exp(logS[:, :-1])], dim=1)
    pmf = S_prev * hazard
    cdf = torch.cumsum(pmf, dim=1).clamp(0, 1)
    return pmf, cdf, logS

def quantile_from_cdf_1d(cdf_1d, q, Tend):
    # cdf_1d: numpy (T,)
    if cdf_1d[-1] < q:
        return Tend
    return int(np.searchsorted(cdf_1d, q) + 1)  # 1..T

def overlap_metrics(pred_L, pred_R, true_L, true_R):
    # pred: [pred_L, pred_R] inclusive
    # true: (L,R] -> [L+1, R] inclusive
    true_L2 = true_L + 1

    inter_L = max(pred_L, true_L2)
    inter_R = min(pred_R, true_R)
    inter = max(0, inter_R - inter_L + 1)

    pred_len = max(1, pred_R - pred_L + 1)
    true_len = max(1, true_R - true_L)  # R-L
    union_L = min(pred_L, true_L2)
    union_R = max(pred_R, true_R)
    union = max(1, union_R - union_L + 1)

    iou = inter / union
    recall = inter / true_len
    precision = inter / pred_len
    return iou, recall, precision

@torch.no_grad()
def eval_metrics_with_overlap(model, loader, Tend, alpha=0.2):
    """
    alpha=0.2 => 중앙 80% 예측구간 [q0.1, q0.9]
    """
    model.eval()
    q_lo = alpha/2
    q_hi = 1 - alpha/2

    hits_all, maes_all, mass_all = [], [], []
    ious, recalls, precs = [], [], []
    hits_int, maes_int, mass_int = [], [], []
    n_int = 0

    for X, L, R, ctype in loader:
        X = X.to(device)
        L_t = L.to(device)
        R_t = R.to(device)

        ctype_np = ctype.cpu().numpy().astype(int)
        L_np = L.cpu().numpy().astype(int)
        R_np = R.cpu().numpy().astype(int)

        hazard = model(X)
        _, cdf, logS = hazard_to_pmf_cdf_logS(hazard)

        # median (CDF가 0.5 못 넘으면 Tend)
        cdf_last = cdf[:, -1]
        arg = (cdf >= 0.5).float().argmax(dim=1) + 1
        median = torch.where(cdf_last >= 0.5, arg, torch.tensor(Tend, device=cdf.device))
        median_np = median.cpu().numpy().astype(int)

        # point coverage & MAE(mid)
        hit = ((median_np > L_np) & (median_np <= R_np)).astype(float)
        mid = np.round((L_np + R_np) / 2.0).astype(int)
        mae = np.abs(median_np - mid).astype(float)

        hits_all.extend(hit.tolist())
        maes_all.extend(mae.tolist())

        # mass-in-interval: S_L - S_R
        idxL = (torch.clamp(L_t, 1, Tend) - 1).long()
        idxR = (torch.clamp(R_t, 1, Tend) - 1).long()
        logS_L = logS.gather(1, idxL.view(-1,1)).squeeze(1)
        logS_R = logS.gather(1, idxR.view(-1,1)).squeeze(1)
        mass = (torch.exp(logS_L) - torch.exp(logS_R)).clamp(min=0.0).cpu().numpy()
        mass_all.extend(mass.tolist())

        # overlap (interval-only)
        cdf_np = cdf.cpu().numpy()
        for b in range(len(L_np)):
            if ctype_np[b] != CTYPE_INTERVAL:
                continue
            n_int += 1
            hits_int.append(hit[b]); maes_int.append(mae[b]); mass_int.append(mass[b])

            pL = quantile_from_cdf_1d(cdf_np[b], q_lo, Tend)
            pR = quantile_from_cdf_1d(cdf_np[b], q_hi, Tend)
            pL = max(1, min(pL, Tend))
            pR = max(1, min(pR, Tend))
            if pL > pR:
                pL, pR = pR, pL

            iou, rec, prec = overlap_metrics(pL, pR, int(L_np[b]), int(R_np[b]))
            ious.append(iou); recalls.append(rec); precs.append(prec)

    out = {
        "point_cov_mean_all": float(np.mean(hits_all)),
        "mae_mid_mean_all": float(np.mean(maes_all)),
        "mass_in_interval_mean_all": float(np.mean(mass_all)),
        "mass_in_interval_median_all": float(np.median(mass_all)),

        "point_cov_mean_interval_only": float(np.mean(hits_int)) if n_int>0 else np.nan,
        "mae_mid_mean_interval_only": float(np.mean(maes_int)) if n_int>0 else np.nan,
        "mass_in_interval_mean_interval_only": float(np.mean(mass_int)) if n_int>0 else np.nan,

        "IoU_mean_interval_only(80%)": float(np.mean(ious)) if n_int>0 else np.nan,
        "Recall_mean_interval_only(80%)": float(np.mean(recalls)) if n_int>0 else np.nan,
        "Precision_mean_interval_only(80%)": float(np.mean(precs)) if n_int>0 else np.nan,
        "N_interval_samples": int(n_int),
    }
    return out

print("OK: functions ready -> run early stopping next.")


OK: functions ready -> run early stopping next.


In [144]:
# 모델 학습 + early stopping

import copy, random
import numpy as np
import torch
from torch.utils.data import DataLoader

SEEDS = [0, 1, 2]

# Early stopping 설정 (Val NLL 기준)
patience = 6
max_epochs = 60
min_delta = 1e-3

# seed별 best_state 저장소
trained_states = []  # 각 원소: {"seed", "best_epoch", "best_val_nll", "state_dict"}

for SEED in SEEDS:
    # seed 고정
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    # loader 재현성(shuffle)
    gen = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=False, generator=gen)
    val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False, drop_last=False)
    test_loader  = DataLoader(test_ds,  batch_size=128, shuffle=False, drop_last=False)  # 여기선 안 씀(학습만)

    # model/init
    torch.manual_seed(SEED)
    model = HazardTransformer(
        d_in=train_samples[0]["X"].shape[1],
        d_model=64,
        nhead=4,
        num_layers=3,
        dropout=0.2
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

    best_val = float("inf")
    best_state = None
    best_epoch = -1
    pat = 0

    for epoch in range(1, max_epochs + 1):
        tr = run_epoch_weighted(model, opt, train_loader, Tend=T, train=True)
        va = eval_nll_model(model, val_loader, Tend=T)
        print(f"[seed {SEED}] epoch {epoch:02d} | train_nll {tr:.4f} | val_nll {va:.4f}")

        if va < best_val - min_delta:
            best_val = float(va)
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                break

    if best_state is None:
        raise ValueError(f"[seed {SEED}] best_state is None")

    trained_states.append({
        "seed": SEED,
        "best_epoch": best_epoch,
        "best_val_nll": best_val,
        "state_dict": best_state
    })

    print(f"[seed {SEED}] DONE | best_epoch={best_epoch} | best_val_nll={best_val:.4f}\n")

print("saved models:", [(d["seed"], d["best_epoch"], d["best_val_nll"]) for d in trained_states])


[seed 0] epoch 01 | train_nll 36.4980 | val_nll 39.8463
[seed 0] epoch 02 | train_nll 27.4061 | val_nll 28.5747
[seed 0] epoch 03 | train_nll 19.6118 | val_nll 19.6040
[seed 0] epoch 04 | train_nll 13.8964 | val_nll 13.5751
[seed 0] epoch 05 | train_nll 10.1025 | val_nll 9.8877
[seed 0] epoch 06 | train_nll 7.6512 | val_nll 7.5563
[seed 0] epoch 07 | train_nll 6.0810 | val_nll 5.9468
[seed 0] epoch 08 | train_nll 4.9197 | val_nll 4.7601
[seed 0] epoch 09 | train_nll 4.0112 | val_nll 3.8464
[seed 0] epoch 10 | train_nll 3.2742 | val_nll 3.1361
[seed 0] epoch 11 | train_nll 2.7228 | val_nll 2.5871
[seed 0] epoch 12 | train_nll 2.2800 | val_nll 2.1730
[seed 0] epoch 13 | train_nll 1.9286 | val_nll 1.8716
[seed 0] epoch 14 | train_nll 1.6865 | val_nll 1.6642
[seed 0] epoch 15 | train_nll 1.5188 | val_nll 1.5352
[seed 0] epoch 16 | train_nll 1.4199 | val_nll 1.4704
[seed 0] epoch 17 | train_nll 1.3819 | val_nll 1.4536
[seed 0] epoch 18 | train_nll 1.3740 | val_nll 1.4456
[seed 0] epoch 19 |

In [145]:
import numpy as np
import torch
from torch.utils.data import DataLoader
import pandas as pd

# 평가용 loader (shuffle 없음, seed 상관없음)
val_loader  = DataLoader(val_ds,  batch_size=128, shuffle=False, drop_last=False)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, drop_last=False)

eval_records = []

for d in trained_states:
    seed = d["seed"]

    # 모델 생성 + state 로드
    model = HazardTransformer(
        d_in=train_samples[0]["X"].shape[1],
        d_model=64,
        nhead=4,
        num_layers=3,
        dropout=0.2
    ).to(device)
    model.load_state_dict(d["state_dict"])

    # NLL
    val_nll  = float(eval_nll_model(model, val_loader, Tend=T))
    test_nll = float(eval_nll_model(model, test_loader, Tend=T))

    # metrics
    val_stats  = eval_metrics_with_overlap(model, val_loader, Tend=T, alpha=0.2)
    test_stats = eval_metrics_with_overlap(model, test_loader, Tend=T, alpha=0.2)

    eval_records.append({
        "seed": seed,
        "best_epoch": d["best_epoch"],
        "best_val_nll": d["best_val_nll"],

        "VAL_NLL": val_nll,
        "TEST_NLL": test_nll,

        # interval-only 핵심 지표(추천)
        "TEST_IoU80(interval_only)": float(test_stats["IoU_mean_interval_only(80%)"]),
        "TEST_Prec80(interval_only)": float(test_stats["Precision_mean_interval_only(80%)"]),
        "TEST_Rec80(interval_only)": float(test_stats["Recall_mean_interval_only(80%)"]),
        "TEST_MAE_int(interval_only)": float(test_stats["mae_mid_mean_interval_only"]),
        "TEST_Mass_int(interval_only)": float(test_stats["mass_in_interval_mean_interval_only"]),
        "TEST_N_int": int(test_stats["N_interval_samples"]),
    })

df = pd.DataFrame(eval_records)
display(df)

def mean_std(col):
    x = df[col].astype(float).to_numpy()
    return x.mean(), x.std(ddof=1)

print("RUN:", RUN )
print("=== mean ± std (seeds) ===")
for col in ["VAL_NLL","TEST_NLL","TEST_IoU80(interval_only)","TEST_Prec80(interval_only)","TEST_Rec80(interval_only)","TEST_MAE_int(interval_only)","TEST_Mass_int(interval_only)"]:
    m, sd = mean_std(col)
    print(f"{col:>12s}: {m:.4f} ± {sd:.4f}")

print("\nTEST interval-only N:", sorted(df["TEST_N_int"].unique()))


,seed,best_epoch,best_val_nll,VAL_NLL,TEST_NLL,TEST_IoU80(interval_only),TEST_Prec80(interval_only),TEST_Rec80(interval_only),TEST_MAE_int(interval_only),TEST_Mass_int(interval_only),TEST_N_int
0,0,60,1.055983,1.055983,1.029609,0.143632,0.170983,0.684366,38.548387,0.275753,31
1,1,60,1.053799,1.053799,0.970630,0.170456,0.207373,0.750859,37.483871,0.317575,31
2,2,59,1.025514,1.025514,1.027042,0.149537,0.204194,0.675567,46.516129,0.292642,31


RUN: 5
=== mean ± std (seeds) ===
     VAL_NLL: 1.0451 ± 0.0170
    TEST_NLL: 1.0091 ± 0.0333
TEST_IoU80(interval_only): 0.1545 ± 0.0141
TEST_Prec80(interval_only): 0.1942 ± 0.0202
TEST_Rec80(interval_only): 0.7036 ± 0.0412
TEST_MAE_int(interval_only): 40.8495 ± 4.9363
TEST_Mass_int(interval_only): 0.2953 ± 0.0210

TEST interval-only N: [np.int64(31)]


### ㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡ

In [146]:
import numpy as np
import torch

@torch.no_grad()
def permutation_importance_features(model, val_loader, Tend, feature_names=None, n_repeats=3, seed=0):
    model.eval()

    base_val_nll = eval_nll_model(model, val_loader, Tend)

    X0, *_ = next(iter(val_loader))
    D = X0.shape[-1]
    if feature_names is None:
        feature_names = [f"f{j}" for j in range(D)]
    assert len(feature_names) == D

    importances = np.zeros(D, dtype=np.float64)
    g = torch.Generator(device="cpu").manual_seed(seed)

    for j in range(D):
        deltas = []
        for _ in range(n_repeats):
            total, n = 0.0, 0
            for X, L, R, ctype in val_loader:
                Xp = X.clone()
                idx = torch.randperm(Xp.size(0), generator=g)

                if Xp.dim() == 2:        # (B, D)
                    Xp[:, j] = Xp[idx, j]
                elif Xp.dim() == 3:      # (B, T, D)
                    Xp[:, :, j] = Xp[idx, :, j]
                else:
                    raise ValueError(f"Unexpected X shape: {Xp.shape}")

                Xp = Xp.to(device); L = L.to(device); R = R.to(device); ctype = ctype.to(device)

                hazard = model(Xp)
                nll_vec = interval_nll_per_sample(hazard, L, R, ctype, Tend=Tend)
                loss = weighted_loss_from_ctype(nll_vec, ctype)

                total += loss.item() * Xp.size(0)
                n += Xp.size(0)

            shuffled_val_nll = total / max(n, 1)
            deltas.append(shuffled_val_nll - base_val_nll)

        importances[j] = float(np.mean(deltas))

    ranked = sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True)
    return base_val_nll, importances, ranked



In [147]:
from torch.utils.data import DataLoader

val_loader_pi = DataLoader(val_ds, batch_size=128, shuffle=False, drop_last=False)


In [148]:
X0, *_ = next(iter(val_loader_pi))
print("D from loader =", X0.shape[-1], " / len(FEATURE_COLS_RUN) =", len(FEATURE_COLS_RUN))
assert X0.shape[-1] == len(FEATURE_COLS_RUN)


D from loader = 18  / len(FEATURE_COLS_RUN) = 18


In [149]:
feature_names = feature_cols  # 네 feature 리스트

all_imps = []
all_bases = []

for d in trained_states:
    SEED = d["seed"]

    # 모델 재생성 + best_state 로드
    model = HazardTransformer(
        d_in=train_samples[0]["X"].shape[1],
        d_model=64,
        nhead=4,
        num_layers=3,
        dropout=0.2
    ).to(device)
    model.load_state_dict(d["state_dict"])
    model.eval()

    base, imps, ranked = permutation_importance_features(
    model, val_loader_pi, Tend=T, feature_names=FEATURE_COLS_RUN, n_repeats=3, seed=SEED
)


    all_bases.append(base)
    all_imps.append(imps)

    print(f"\n[seed {SEED}] base_val_nll={base:.4f}  TOP10:")
    for name, imp in ranked[:10]:
        print(f"  {name:30s}  ΔNLL={imp:.6f}")



[seed 0] base_val_nll=1.0560  TOP10:
  좌표-위도                           ΔNLL=0.208218
  n_obs                           ΔNLL=0.113172
  days_since_flowering            ΔNLL=0.101831
  days_since_growing_start        ΔNLL=0.077473
  tmean_7d_mean                   ΔNLL=0.064782
  좌표-경도                           ΔNLL=0.063762
  trange_7d_mean                  ΔNLL=0.047889
  max_gap                         ΔNLL=0.042050
  first_obs_season                ΔNLL=0.037484
  rain_7d_days                    ΔNLL=0.032921

[seed 1] base_val_nll=1.0538  TOP10:
  좌표-위도                           ΔNLL=0.281944
  좌표-경도                           ΔNLL=0.218802
  n_obs                           ΔNLL=0.080143
  DD10_7d_sum                     ΔNLL=0.053925
  is_growing                      ΔNLL=0.037453
  rain_7d_days                    ΔNLL=0.033631
  first_obs_season                ΔNLL=0.033581
  days_since_flowering            ΔNLL=0.030226
  max_gap                         ΔNLL=0.030221
  trange    

In [150]:
mean_imps = np.mean(np.stack(all_imps, axis=0), axis=0)
ranked_mean = sorted(zip(feature_names, mean_imps), key=lambda x: x[1], reverse=True)

print("\n=== MEAN importance TOP20 (seeds) ===")
for name, imp in ranked_mean[:20]:
    print(f"{name:30s}  meanΔNLL={imp:.6f}")

TOPK = [name for name, _ in ranked_mean[:20]]
print("\nTOPK =", TOPK)



=== MEAN importance TOP20 (seeds) ===
좌표-위도                           meanΔNLL=0.230900
좌표-경도                           meanΔNLL=0.158531
n_obs                           meanΔNLL=0.115990
days_since_flowering            meanΔNLL=0.070180
trange_7d_mean                  meanΔNLL=0.054245
rain_7d_days                    meanΔNLL=0.048917
max_gap                         meanΔNLL=0.038650
first_obs_season                meanΔNLL=0.037865
is_growing                      meanΔNLL=0.036634
trange                          meanΔNLL=0.036614
days_since_growing_start        meanΔNLL=0.033529
tmean_7d_mean                   meanΔNLL=0.022672
DD10_7d_sum                     meanΔNLL=0.021945
tmin_7d_min                     meanΔNLL=0.004492
rain_7d_sum                     meanΔNLL=0.003764
rain_14d_sum                    meanΔNLL=0.003759
GDD10_since_db                  meanΔNLL=-0.000511
tmax_7d_max                     meanΔNLL=-0.001081

TOPK = ['좌표-위도', '좌표-경도', 'n_obs', 'days_since_flowering',

In [151]:
# ===== Cell 1) SFS 후보 고정 (A: meta 포함, TOPK 그대로) =====
CANDIDATES = TOPK[:]   # 18개
print("num candidates:", len(CANDIDATES))
print(CANDIDATES)


num candidates: 18
['좌표-위도', '좌표-경도', 'n_obs', 'days_since_flowering', 'trange_7d_mean', 'rain_7d_days', 'max_gap', 'first_obs_season', 'is_growing', 'trange', 'days_since_growing_start', 'tmean_7d_mean', 'DD10_7d_sum', 'tmin_7d_min', 'rain_7d_sum', 'rain_14d_sum', 'GDD10_since_db', 'tmax_7d_max']


In [152]:
# ===== Cell 2) trial feature_cols -> train/val loaders 만드는 함수 =====
import torch
from torch.utils.data import DataLoader

def make_loaders_for_features(trial_cols, SEED):
    # 1) samples 생성 (trial_cols 기준)
    samples_trial = build_samples_season(train_df_season, trial_cols, DOY_START, DOY_END)

    # 2) split (site 기준) - test는 SFS에 필요 없어서 버림
    train_s, val_s, _ = split_by_site(samples_trial, val_frac=0.1, test_frac=0.1, seed=SEED)

    # 3) norm stats (train만)
    x_mean, x_std = compute_norm_stats(train_s)

    # 4) Dataset / Loader
    train_ds = IntervalEventDataset(train_s, x_mean, x_std)
    val_ds   = IntervalEventDataset(val_s,   x_mean, x_std)

    gen = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=False, generator=gen)
    val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False, drop_last=False)
    return train_loader, val_loader


In [153]:
# ===== Cell 3) SFS 본체(빠른 early stopping) =====
import copy, random
import numpy as np
import torch

def sfs_topk(
    candidates,
    SEED=0,
    max_k=12,
    max_epochs_fs=25,
    patience_fs=5,
    min_delta_fs=1e-3,
    min_gain=1e-4
):
    selected = []
    best_val = float("inf")

    for step in range(1, max_k + 1):
        best_feat = None
        best_val_step = best_val

        for f in candidates:
            if f in selected:
                continue

            trial = selected + [f]

            # seed 고정
            random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

            train_loader, val_loader = make_loaders_for_features(trial, SEED)

            # 모델 init
            model = HazardTransformer(
                d_in=len(trial),
                d_model=64,
                nhead=4,
                num_layers=3,
                dropout=0.2
            ).to(device)
            opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

            best_local = float("inf")
            pat = 0

            for epoch in range(1, max_epochs_fs + 1):
                _ = run_epoch_weighted(model, opt, train_loader, Tend=T, train=True)
                va = eval_nll_model(model, val_loader, Tend=T)

                if va < best_local - min_delta_fs:
                    best_local = float(va)
                    pat = 0
                else:
                    pat += 1
                    if pat >= patience_fs:
                        break

            if best_local < best_val_step:
                best_val_step = best_local
                best_feat = f

        gain = best_val - best_val_step
        if best_feat is None or gain < min_gain:
            print(f"[SFS] stop at step {step} | gain={gain:.6f}")
            break

        selected.append(best_feat)
        best_val = best_val_step
        print(f"[SFS] step {step:02d}: +{best_feat:25s} val_nll={best_val:.5f} gain={gain:.6f}")

    return selected, best_val


In [154]:
# ===== Cell 4) 실행 =====
best_feats, best_val = sfs_topk(CANDIDATES, SEED=0, max_k=12)
print("\nBEST_FEATS =", best_feats)
print("BEST_VAL_NLL =", best_val)


[SFS] step 01: +n_obs                     val_nll=1.21094 gain=inf
[SFS] step 02: +trange_7d_mean            val_nll=1.06525 gain=0.145691
[SFS] step 03: +first_obs_season          val_nll=1.05167 gain=0.013581
[SFS] step 04: +tmin_7d_min               val_nll=1.03359 gain=0.018076
[SFS] stop at step 5 | gain=0.000000

BEST_FEATS = ['n_obs', 'trange_7d_mean', 'first_obs_season', 'tmin_7d_min']
BEST_VAL_NLL = 1.0335906744003296
